# Portfolio Shared Constraints

This tutorial shows what changed in `v0.5.0`: the primary runtime object is now a **site** with one or more **battery assets**. We compare a single-asset Belgian reserve run against a two-asset portfolio behind a shared POI.

Learning goals:
- understand the `site` + `assets` config model
- run a portfolio `da_plus_fcr` benchmark
- inspect site-level vs asset-level artifacts
- quantify portfolio uplift against a single-asset baseline


## Step 1 - Load single-asset and portfolio configs

Both configs use the same market data, but the portfolio case has two heterogeneous batteries sharing one POI.

In [ ]:
from __future__ import annotations

import tempfile
from pathlib import Path

import pandas as pd

from euroflex_bess_lab import compare_runs, load_config, run_walk_forward
from euroflex_bess_lab.analytics.reporting import load_report_summary

PROJECT_ROOT = Path.cwd()
CONFIG_PATH = PROJECT_ROOT / "examples" / "configs" / "canonical" / "belgium_full_stack.yaml"

portfolio_config = load_config(CONFIG_PATH)
single_payload = portfolio_config.model_dump(mode="json")
single_payload["run_name"] = "belgium-single-asset-reference"
single_payload["site"]["id"] = "belgium-single-asset-site"
single_payload["assets"] = [single_payload["assets"][0]]
single_payload["site"]["poi_import_limit_mw"] = 1.0
single_payload["site"]["poi_export_limit_mw"] = 1.0
single_config = type(portfolio_config).model_validate(single_payload)

tmp_root = Path(tempfile.mkdtemp(prefix="euroflex-v10-portfolio-"))
single_config.artifacts.root_dir = tmp_root / "single"
portfolio_config.artifacts.root_dir = tmp_root / "portfolio"

tmp_root

## Step 2 - Run both benchmarks

The single-asset run provides the reference. The portfolio run decides how to allocate energy and reserve commitments across both batteries while respecting the shared site POI.

In [ ]:
single_result = run_walk_forward(single_config)
portfolio_result = run_walk_forward(portfolio_config)

summary = pd.DataFrame(
    [
        load_report_summary(single_result.output_dir),
        load_report_summary(portfolio_result.output_dir),
    ]
)[
    [
        "run_id",
        "run_scope",
        "asset_count",
        "total_pnl_eur",
        "reserve_capacity_revenue_eur",
        "throughput_mwh",
        "reserved_capacity_mw_avg",
        "poi_import_limit_mw",
        "poi_export_limit_mw",
    ]
]
summary

## Step 3 - Inspect site-level dispatch

`site_dispatch.parquet` is the stable site-level artifact. It is the right place to inspect POI utilization, total reserved MW, and site-level reason codes.

In [ ]:
portfolio_result.site_dispatch[
    [
        "timestamp_local",
        "charge_mw",
        "discharge_mw",
        "fcr_reserved_mw",
        "reserve_headroom_up_mw",
        "reserve_headroom_down_mw",
        "reason_code",
    ]
].head(8)

## Step 4 - Inspect asset-level allocation

`asset_dispatch.parquet` shows how the site-level plan is split across batteries. This is where heterogeneous specs, headroom, and degradation assumptions become visible.

In [ ]:
portfolio_result.asset_dispatch[
    [
        "timestamp_local",
        "asset_id",
        "charge_mw",
        "discharge_mw",
        "soc_mwh",
        "fcr_reserved_mw",
        "availability_factor",
        "reason_code",
    ]
].head(12)

## Step 5 - Compare runs the same way the CLI does

The comparison layer can consume single-asset and portfolio runs without any special casing.

In [ ]:
comparison_dir = compare_runs(
    [single_result.output_dir, portfolio_result.output_dir],
    tmp_root / "comparison",
    group_by="workflow",
)

pd.read_csv(comparison_dir / "comparison.csv")[
    [
        "run_id",
        "run_scope",
        "total_pnl_eur",
        "portfolio_uplift_vs_single_asset_eur",
        "reserve_share_of_total_revenue",
    ]
]

## Takeaways

- `schema_version: 4` makes site constraints explicit.
- Portfolio runs write both site and asset dispatch artifacts.
- Shared-POI optimization creates a more realistic commercial screening workflow than a single battery in isolation.
